# Real-Time MEV Front-Running Detection Model

This notebook builds a machine learning model to detect MEV front-running auctions IN ACTION using mempool data.

Key difference from FlashBoys: We predict auctions BEFORE they complete, giving 1-5 second lead time.

Data source: Flashbots Mempool Dumpster (https://mempool-dumpster.flashbots.net)

In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
import requests
from datetime import datetime, timedelta
import pyarrow.parquet as pq
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## 1. Load Pre-Downloaded Mempool Data

In [2]:
DATA_DIR = Path("data/mempool")

def get_downloaded_files():
    parquet_files = sorted(DATA_DIR.glob("*.parquet"))
    
    if not parquet_files:
        logger.error(f"No parquet files found in {DATA_DIR}")
        logger.info("Run: python download_mempool_historical.py")
        return []
    
    logger.info(f"Found {len(parquet_files)} parquet files:")
    for f in parquet_files:
        size_mb = f.stat().st_size / 1e6
        logger.info(f"  {f.name} ({size_mb:.1f} MB)")
    
    return parquet_files

downloaded_files = get_downloaded_files()

if downloaded_files:
    date_range = f"{downloaded_files[0].stem} to {downloaded_files[-1].stem}"
    logger.info(f"Date range: {date_range}")
    logger.info(f"Ready to load {len(downloaded_files)} files")

2025-11-08 18:28:00,411 - INFO - Found 8 parquet files:
2025-11-08 18:28:00,412 - INFO -   2025-10-31.parquet (4772.9 MB)
2025-11-08 18:28:00,412 - INFO -   2025-11-01.parquet (4750.3 MB)
2025-11-08 18:28:00,412 - INFO -   2025-11-02.parquet (4731.8 MB)
2025-11-08 18:28:00,412 - INFO -   2025-11-03.parquet (5190.7 MB)
2025-11-08 18:28:00,413 - INFO -   2025-11-04.parquet (5785.5 MB)
2025-11-08 18:28:00,413 - INFO -   2025-11-05.parquet (4620.6 MB)
2025-11-08 18:28:00,413 - INFO -   2025-11-06.parquet (4676.1 MB)
2025-11-08 18:28:00,413 - INFO -   2025-11-07.parquet (5632.1 MB)
2025-11-08 18:28:00,413 - INFO - Date range: 2025-10-31 to 2025-11-07
2025-11-08 18:28:00,414 - INFO - Ready to load 8 files
2025-11-08 18:28:00,412 - INFO -   2025-10-31.parquet (4772.9 MB)
2025-11-08 18:28:00,412 - INFO -   2025-11-01.parquet (4750.3 MB)
2025-11-08 18:28:00,412 - INFO -   2025-11-02.parquet (4731.8 MB)
2025-11-08 18:28:00,412 - INFO -   2025-11-03.parquet (5190.7 MB)
2025-11-08 18:28:00,413 - I

## 2. Load and Parse Mempool Data

In [3]:
def load_parquet_file(filepath):
    logger.info(f"Loading: {filepath.name}")
    
    df = pq.read_table(filepath).to_pandas()
    
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['includedBlockTimestamp'] = pd.to_datetime(df['includedBlockTimestamp'])
    
    df['gasPrice'] = pd.to_numeric(df['gasPrice'], errors='coerce')
    df['gas'] = pd.to_numeric(df['gas'], errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['nonce'] = pd.to_numeric(df['nonce'], errors='coerce')
    
    logger.info(f"Loaded {len(df):,} transactions")
    return df

def load_all_mempool_data(files):
    all_data = []
    
    for filepath in files:
        df = load_parquet_file(filepath)
        all_data.append(df)
    
    combined = pd.concat(all_data, ignore_index=True)
    combined = combined.sort_values('timestamp').reset_index(drop=True)
    
    logger.info(f"Total transactions loaded: {len(combined):,}")
    return combined

mempool_df = load_all_mempool_data(downloaded_files)
logger.info(f"Date range: {mempool_df['timestamp'].min()} to {mempool_df['timestamp'].max()}")
logger.info(f"Columns: {list(mempool_df.columns)}")

2025-11-08 18:28:00,418 - INFO - Loading: 2025-10-31.parquet
2025-11-08 18:28:10,543 - INFO - Loaded 1,114,285 transactions
2025-11-08 18:28:10,546 - INFO - Loading: 2025-11-01.parquet
2025-11-08 18:28:10,543 - INFO - Loaded 1,114,285 transactions
2025-11-08 18:28:10,546 - INFO - Loading: 2025-11-01.parquet
2025-11-08 18:28:18,124 - INFO - Loaded 945,695 transactions
2025-11-08 18:28:18,125 - INFO - Loading: 2025-11-02.parquet
2025-11-08 18:28:18,124 - INFO - Loaded 945,695 transactions
2025-11-08 18:28:18,125 - INFO - Loading: 2025-11-02.parquet
2025-11-08 18:28:25,520 - INFO - Loaded 1,048,641 transactions
2025-11-08 18:28:25,521 - INFO - Loading: 2025-11-03.parquet
2025-11-08 18:28:25,520 - INFO - Loaded 1,048,641 transactions
2025-11-08 18:28:25,521 - INFO - Loading: 2025-11-03.parquet
2025-11-08 18:28:34,104 - INFO - Loaded 1,125,550 transactions
2025-11-08 18:28:34,105 - INFO - Loading: 2025-11-04.parquet
2025-11-08 18:28:34,104 - INFO - Loaded 1,125,550 transactions
2025-11-08 1

## 3. FlashBoys Labeling: Detect Gas Auctions (Retrospective)

In [4]:
class FlashBoysLabeler:
    def __init__(self, time_window=3.0):
        self.time_window = time_window
    
    def detect_gas_auctions(self, df):
        logger.info("Running FlashBoys auction detection...")
        
        df = df.sort_values('timestamp').reset_index(drop=True)
        df['is_mev_auction'] = 0
        df['auction_id'] = -1
        
        auction_id = 0
        i = 0
        
        while i < len(df) - 1:
            auction_txs = [i]
            bidders = {(df.loc[i, 'from'], df.loc[i, 'nonce'])}
            
            j = i + 1
            while j < len(df):
                time_diff = (df.loc[j, 'timestamp'] - df.loc[auction_txs[-1], 'timestamp']).total_seconds()
                
                if time_diff > self.time_window:
                    break
                
                bidder = (df.loc[j, 'from'], df.loc[j, 'nonce'])
                
                if bidder not in bidders:
                    auction_txs.append(j)
                    bidders.add(bidder)
                
                j += 1
            
            if len(auction_txs) >= 2:
                gas_prices = df.loc[auction_txs, 'gasPrice'].values
                price_increases = sum(1 for k in range(len(gas_prices)-1) if gas_prices[k+1] > gas_prices[k])
                escalation_ratio = price_increases / (len(gas_prices) - 1) if len(gas_prices) > 1 else 0
                
                if escalation_ratio > 0.3:
                    df.loc[auction_txs, 'is_mev_auction'] = 1
                    df.loc[auction_txs, 'auction_id'] = auction_id
                    auction_id += 1
            
            i = j if j > i + 1 else i + 1
        
        num_auctions = df['auction_id'].max() + 1
        num_mev_txs = (df['is_mev_auction'] == 1).sum()
        
        logger.info(f"Detected {num_auctions:,} gas auctions")
        logger.info(f"MEV transactions: {num_mev_txs:,} ({100*num_mev_txs/len(df):.2f}%)")
        
        return df

labeler = FlashBoysLabeler(time_window=3.0)
mempool_df = labeler.detect_gas_auctions(mempool_df)

logger.info(f"Label distribution: {mempool_df['is_mev_auction'].value_counts().to_dict()}")

2025-11-08 18:29:41,235 - INFO - Running FlashBoys auction detection...
2025-11-08 18:31:23,171 - INFO - Detected 11 gas auctions
2025-11-08 18:31:23,171 - INFO - Detected 11 gas auctions
2025-11-08 18:31:23,173 - INFO - MEV transactions: 8,252,783 (94.04%)
2025-11-08 18:31:23,622 - INFO - Label distribution: {1: 8252783, 0: 522893}
2025-11-08 18:31:23,173 - INFO - MEV transactions: 8,252,783 (94.04%)
2025-11-08 18:31:23,622 - INFO - Label distribution: {1: 8252783, 0: 522893}


## 4. Feature Engineering: Extract Predictive Signals

In [5]:
class MempoolFeatureExtractor:
    def __init__(self, lookback_window=20):
        self.lookback_window = lookback_window
    
    def extract_features(self, df):
        logger.info("Extracting temporal features...")
        
        features = []
        labels = []
        
        for idx in tqdm(range(self.lookback_window, len(df)), desc="Feature extraction"):
            window_start = max(0, idx - self.lookback_window)
            window = df.iloc[window_start:idx]
            
            if len(window) == 0:
                continue
            
            gas_prices = window['gasPrice'].values
            timestamps = window['timestamp'].values
            
            time_span = (timestamps[-1] - timestamps[0]) / np.timedelta64(1, 's') if len(timestamps) > 1 else 0
            
            feat = {
                'tx_index': idx,
                'gas_price': df.loc[idx, 'gasPrice'],
                'gas_limit': df.loc[idx, 'gas'],
                'tx_value': df.loc[idx, 'value'],
                'recent_gas_mean': np.mean(gas_prices),
                'recent_gas_std': np.std(gas_prices),
                'recent_gas_median': np.median(gas_prices),
                'recent_gas_max': np.max(gas_prices),
                'recent_gas_min': np.min(gas_prices),
                'gas_vs_mean': df.loc[idx, 'gasPrice'] / (np.mean(gas_prices) + 1e-9),
                'gas_vs_median': df.loc[idx, 'gasPrice'] / (np.median(gas_prices) + 1e-9),
                'gas_vs_max': df.loc[idx, 'gasPrice'] / (np.max(gas_prices) + 1e-9),
                'tx_density': len(window) / (time_span + 1),
                'unique_senders': window['from'].nunique(),
                'unique_receivers': window['to'].nunique(),
            }
            
            if len(gas_prices) >= 3:
                recent_3 = gas_prices[-3:]
                price_changes = np.diff(recent_3)
                feat['gas_momentum'] = np.mean(price_changes) if len(price_changes) > 0 else 0
                feat['gas_acceleration'] = price_changes[-1] - price_changes[0] if len(price_changes) >= 2 else 0
                
                increasing = sum(1 for x in price_changes if x > 0)
                feat['price_increase_ratio'] = increasing / len(price_changes) if len(price_changes) > 0 else 0
            else:
                feat['gas_momentum'] = 0
                feat['gas_acceleration'] = 0
                feat['price_increase_ratio'] = 0
            
            threshold_high = np.percentile(gas_prices, 75)
            feat['high_gas_count'] = (gas_prices > threshold_high).sum()
            feat['high_gas_ratio'] = feat['high_gas_count'] / len(gas_prices)
            
            sender_nonces = window.groupby('from')['nonce'].apply(list)
            feat['sender_nonce_gaps'] = sum(
                sum(1 for i in range(len(nonces)-1) if nonces[i+1] != nonces[i] + 1)
                for nonces in sender_nonces if len(nonces) > 1
            )
            
            features.append(feat)
            labels.append(df.loc[idx, 'is_mev_auction'])
        
        feature_df = pd.DataFrame(features)
        feature_df['label'] = labels
        
        logger.info(f"Extracted {len(feature_df):,} feature vectors")
        logger.info(f"Feature columns: {len(feature_df.columns)}")
        
        return feature_df

extractor = MempoolFeatureExtractor(lookback_window=20)

sample_size = min(500000, len(mempool_df))
sample_df = mempool_df.iloc[:sample_size].copy()
logger.info(f"Processing sample of {len(sample_df):,} transactions")

feature_df = extractor.extract_features(sample_df)

logger.info(f"Feature shape: {feature_df.shape}")
logger.info(f"Label distribution: {feature_df['label'].value_counts().to_dict()}")

2025-11-08 18:31:23,656 - INFO - Processing sample of 500,000 transactions
2025-11-08 18:31:23,656 - INFO - Extracting temporal features...
2025-11-08 18:31:23,656 - INFO - Extracting temporal features...
Feature extraction: 100%|██████████| 499980/499980 [03:00<00:00, 2769.10it/s]

2025-11-08 18:34:26,040 - INFO - Extracted 499,980 feature vectors
2025-11-08 18:34:26,040 - INFO - Extracted 499,980 feature vectors
2025-11-08 18:34:26,040 - INFO - Feature columns: 22
2025-11-08 18:34:26,107 - INFO - Feature shape: (499980, 22)
2025-11-08 18:34:26,109 - INFO - Label distribution: {1: 481567, 0: 18413}
2025-11-08 18:34:26,040 - INFO - Feature columns: 22
2025-11-08 18:34:26,107 - INFO - Feature shape: (499980, 22)
2025-11-08 18:34:26,109 - INFO - Label distribution: {1: 481567, 0: 18413}


## 5. Train/Test Split: Temporal Validation

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = [col for col in feature_df.columns if col not in ['label', 'tx_index']]

X = feature_df[feature_cols].fillna(0)
y = feature_df['label']

split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

logger.info(f"Train set: {len(X_train):,} samples")
logger.info(f"Test set: {len(X_test):,} samples")
logger.info(f"Train MEV ratio: {y_train.mean():.4f}")
logger.info(f"Test MEV ratio: {y_test.mean():.4f}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

logger.info("Feature scaling complete")

2025-11-08 18:34:26,781 - INFO - Train set: 399,984 samples
2025-11-08 18:34:26,781 - INFO - Test set: 99,996 samples
2025-11-08 18:34:26,782 - INFO - Train MEV ratio: 0.9619
2025-11-08 18:34:26,782 - INFO - Test MEV ratio: 0.9681
2025-11-08 18:34:26,781 - INFO - Test set: 99,996 samples
2025-11-08 18:34:26,782 - INFO - Train MEV ratio: 0.9619
2025-11-08 18:34:26,782 - INFO - Test MEV ratio: 0.9681
2025-11-08 18:34:26,838 - INFO - Feature scaling complete
2025-11-08 18:34:26,838 - INFO - Feature scaling complete


## 6. Train ML Model: LightGBM Classifier

In [8]:
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

logger.info("Training LightGBM model...")

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum() if (y_train == 1).sum() > 0 else 1

model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model.fit(
    X_train_scaled, 
    y_train,
    eval_set=[(X_test_scaled, y_test)],
    callbacks=[lgb.log_evaluation(period=50)]
)

logger.info("Training complete")

2025-11-08 18:36:09,840 - INFO - Training LightGBM model...


[50]	valid_0's binary_logloss: 0.167462
[100]	valid_0's binary_logloss: 0.171868
[100]	valid_0's binary_logloss: 0.171868
[150]	valid_0's binary_logloss: 0.166683
[150]	valid_0's binary_logloss: 0.166683


2025-11-08 18:36:11,700 - INFO - Training complete


[200]	valid_0's binary_logloss: 0.161119


## 7. Model Evaluation: Prediction Performance

In [9]:
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

logger.info("\nClassification Report:")
logger.info(f"\n{classification_report(y_test, y_pred)}")

logger.info("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
logger.info(f"\n{cm}")

roc_auc = roc_auc_score(y_test, y_pred_proba)
logger.info(f"\nROC AUC Score: {roc_auc:.4f}")

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

logger.info("\nTop 10 Most Important Features:")
for idx, row in feature_importance.head(10).iterrows():
    logger.info(f"  {row['feature']}: {row['importance']:.4f}")

2025-11-08 18:37:03,305 - INFO - 
Classification Report:
2025-11-08 18:37:03,319 - INFO - 
              precision    recall  f1-score   support

           0       0.28      0.88      0.42      3188
           1       1.00      0.92      0.96     96808

    accuracy                           0.92     99996
   macro avg       0.64      0.90      0.69     99996
weighted avg       0.97      0.92      0.94     99996

2025-11-08 18:37:03,319 - INFO - 
Confusion Matrix:
2025-11-08 18:37:03,323 - INFO - 
[[ 2792   396]
 [ 7282 89526]]
2025-11-08 18:37:03,319 - INFO - 
              precision    recall  f1-score   support

           0       0.28      0.88      0.42      3188
           1       1.00      0.92      0.96     96808

    accuracy                           0.92     99996
   macro avg       0.64      0.90      0.69     99996
weighted avg       0.97      0.92      0.94     99996

2025-11-08 18:37:03,319 - INFO - 
Confusion Matrix:
2025-11-08 18:37:03,323 - INFO - 
[[ 2792   396]
 [ 

## 8. Lead Time Analysis: How Early Can We Predict?

In [ ]:
def calculate_prediction_lead_time(feature_df, predictions, probabilities, threshold=0.5):
    results = feature_df.copy()
    results['prediction'] = predictions
    results['probability'] = probabilities
    results['predicted_mev'] = (probabilities >= threshold).astype(int)
    
    lead_times = []
    
    auction_groups = sample_df.iloc[results.index].groupby('auction_id')
    
    for auction_id, auction_df in auction_groups:
        if auction_id == -1:
            continue
        
        auction_indices = auction_df.index
        auction_results = results[results.index.isin(auction_indices)]
        
        first_prediction_idx = auction_results[auction_results['predicted_mev'] == 1].index
        
        if len(first_prediction_idx) > 0:
            first_pred = first_prediction_idx[0]
            first_actual = auction_indices[0]
            
            lead_txs = first_pred - first_actual
            
            time_first_pred = sample_df.loc[first_pred, 'timestamp']
            time_first_actual = sample_df.loc[first_actual, 'timestamp']
            lead_secs = (time_first_pred - time_first_actual).total_seconds()
            
            lead_times.append({
                'auction_id': auction_id,
                'lead_transactions': lead_txs,
                'lead_seconds': lead_secs,
                'auction_size': len(auction_indices)
            })
    
    lead_df = pd.DataFrame(lead_times)
    
    if len(lead_df) > 0:
        logger.info("\nPrediction Lead Time Analysis:")
        logger.info(f"  Auctions detected early: {len(lead_df):,}")
        logger.info(f"  Mean lead time: {lead_df['lead_seconds'].mean():.2f} seconds")
        logger.info(f"  Median lead time: {lead_df['lead_seconds'].median():.2f} seconds")
        logger.info(f"  Mean lead (txs): {lead_df['lead_transactions'].mean():.1f} transactions")
        logger.info(f"  Negative lead times (late): {(lead_df['lead_seconds'] < 0).sum()}")
        logger.info(f"  Positive lead times (early): {(lead_df['lead_seconds'] > 0).sum()}")
    
    return lead_df

test_predictions = y_pred
test_probabilities = y_pred_proba
test_feature_df = feature_df.iloc[split_idx:]

lead_times = calculate_prediction_lead_time(
    test_feature_df, 
    test_predictions, 
    test_probabilities,
    threshold=0.5
)

## 9. Visualization: Model Performance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

top_features = feature_importance.head(15)
axes[0, 0].barh(top_features['feature'], top_features['importance'])
axes[0, 0].set_xlabel('Importance')
axes[0, 0].set_title('Top 15 Feature Importances')
axes[0, 0].invert_yaxis()

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[0, 1].plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve')
axes[0, 1].legend()

cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', ax=axes[1, 0])
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_title('Confusion Matrix (Normalized)')

if len(lead_times) > 0:
    axes[1, 1].hist(lead_times['lead_seconds'], bins=50, edgecolor='black')
    axes[1, 1].axvline(0, color='red', linestyle='--', label='FlashBoys Detection Time')
    axes[1, 1].set_xlabel('Lead Time (seconds)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Prediction Lead Time Distribution')
    axes[1, 1].legend()

plt.tight_layout()
plt.savefig('model_performance.png', dpi=300, bbox_inches='tight')
logger.info("Saved visualization: model_performance.png")
plt.show()

## 10. Save Model and Results

In [ ]:
import joblib

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "mev_detector_lightgbm.pkl"
scaler_path = MODEL_DIR / "feature_scaler.pkl"
feature_path = MODEL_DIR / "feature_columns.txt"

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)
with open(feature_path, 'w') as f:
    f.write('\n'.join(feature_cols))

logger.info(f"Model saved: {model_path}")
logger.info(f"Scaler saved: {scaler_path}")
logger.info(f"Features saved: {feature_path}")

results_path = MODEL_DIR / "evaluation_results.txt"
with open(results_path, 'w') as f:
    f.write("MEV Front-Running Detection Model Results\n")
    f.write("="*50 + "\n\n")
    f.write(f"Training samples: {len(X_train):,}\n")
    f.write(f"Test samples: {len(X_test):,}\n")
    f.write(f"ROC AUC: {roc_auc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_test, y_pred))
    f.write("\n\nTop 10 Features:\n")
    for idx, row in feature_importance.head(10).iterrows():
        f.write(f"  {row['feature']}: {row['importance']:.4f}\n")
    
    if len(lead_times) > 0:
        f.write("\n\nLead Time Analysis:\n")
        f.write(f"  Mean lead time: {lead_times['lead_seconds'].mean():.2f}s\n")
        f.write(f"  Median lead time: {lead_times['lead_seconds'].median():.2f}s\n")
        f.write(f"  Early detections: {(lead_times['lead_seconds'] > 0).sum()}\n")

logger.info(f"Results saved: {results_path}")
logger.info("\nModel training complete!")

## 11. Real-Time Inference Example

In [ ]:
class RealtimeMEVDetector:
    def __init__(self, model, scaler, feature_cols, lookback=20):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.lookback = lookback
        self.tx_buffer = []
    
    def add_transaction(self, tx):
        self.tx_buffer.append(tx)
        if len(self.tx_buffer) > self.lookback:
            self.tx_buffer.pop(0)
    
    def predict(self):
        if len(self.tx_buffer) < 2:
            return 0.0, False
        
        window = pd.DataFrame(self.tx_buffer)
        gas_prices = window['gasPrice'].values
        
        features = {
            'gas_price': self.tx_buffer[-1]['gasPrice'],
            'gas_limit': self.tx_buffer[-1]['gas'],
            'tx_value': self.tx_buffer[-1]['value'],
            'recent_gas_mean': np.mean(gas_prices),
            'recent_gas_std': np.std(gas_prices),
            'recent_gas_median': np.median(gas_prices),
            'recent_gas_max': np.max(gas_prices),
            'recent_gas_min': np.min(gas_prices),
            'gas_vs_mean': gas_prices[-1] / (np.mean(gas_prices) + 1e-9),
            'gas_vs_median': gas_prices[-1] / (np.median(gas_prices) + 1e-9),
            'gas_vs_max': gas_prices[-1] / (np.max(gas_prices) + 1e-9),
            'tx_density': len(window),
            'unique_senders': window['from'].nunique(),
            'unique_receivers': window['to'].nunique(),
        }
        
        if len(gas_prices) >= 3:
            recent_3 = gas_prices[-3:]
            price_changes = np.diff(recent_3)
            features['gas_momentum'] = np.mean(price_changes)
            features['gas_acceleration'] = price_changes[-1] - price_changes[0] if len(price_changes) >= 2 else 0
            increasing = sum(1 for x in price_changes if x > 0)
            features['price_increase_ratio'] = increasing / len(price_changes)
        else:
            features['gas_momentum'] = 0
            features['gas_acceleration'] = 0
            features['price_increase_ratio'] = 0
        
        threshold_high = np.percentile(gas_prices, 75)
        features['high_gas_count'] = (gas_prices > threshold_high).sum()
        features['high_gas_ratio'] = features['high_gas_count'] / len(gas_prices)
        
        sender_nonces = window.groupby('from')['nonce'].apply(list)
        features['sender_nonce_gaps'] = sum(
            sum(1 for i in range(len(nonces)-1) if nonces[i+1] != nonces[i] + 1)
            for nonces in sender_nonces if len(nonces) > 1
        )
        
        X = pd.DataFrame([features])[self.feature_cols].fillna(0)
        X_scaled = self.scaler.transform(X)
        
        probability = self.model.predict_proba(X_scaled)[0, 1]
        prediction = probability >= 0.5
        
        return probability, prediction

detector = RealtimeMEVDetector(model, scaler, feature_cols, lookback=20)

logger.info("\nSimulating real-time detection on test data...")
test_sample = sample_df.iloc[split_idx:split_idx+100]

for idx, row in test_sample.iterrows():
    tx = {
        'gasPrice': row['gasPrice'],
        'gas': row['gas'],
        'value': row['value'],
        'from': row['from'],
        'to': row['to'],
        'nonce': row['nonce']
    }
    
    detector.add_transaction(tx)
    prob, pred = detector.predict()
    
    if pred:
        logger.info(f"MEV Alert: Transaction {idx} - Probability: {prob:.2%}")

logger.info("\nReal-time detector ready for deployment")